# Embedding + Retrieval — The Unofficial Guide (Milestone 4)

This notebook demonstrates the **Embedding + Vector Store** and **Retrieval** stages of the RAG pipeline.

The heavy lifting lives in [`embed_index.py`](embed_index.py); here we just drive it:

1. Load the local embedding model `all-MiniLM-L6-v2` (sentence-transformers, no API key, no rate limits).
2. Token-split reviews (250 tokens, 18% overlap) and embed them into a persistent ChromaDB store with source metadata.
3. Inspect chunk analytics.
4. Run the 5 evaluation questions from `planning.md` through `retrieve()` (top-k = 40).
5. Try an ad-hoc query.

In [1]:
from embed_index import (
    build_index, retrieve, get_model, get_collection, load_reviews, chunk_text,
)

model = get_model()
print('Loaded embedding model:', model.__class__.__name__, '/ all-MiniLM-L6-v2')
print('Max sequence length (tokens):', model.max_seq_length)

C:\Users\Placido Garay\ai201-project1-unofficial-guide-starter\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded embedding model: SentenceTransformer / all-MiniLM-L6-v2
Max sequence length (tokens): 256


## 1. Build the vector store

Chunk every review, embed each chunk, and store it in ChromaDB (cosine space). Re-running rebuilds the collection from scratch.

In [2]:
collection = build_index()

45 reviews -> 45 chunks. Embedding with all-MiniLM-L6-v2 ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:  50%|█████     | 1/2 [00:00<00:00,  2.22it/s]

Batches: 100%|██████████| 2/2 [00:00<00:00,  3.63it/s]

Stored 45 chunks in ChromaDB collection 'professor_reviews' at ./chroma_db


## 2. Chunk analytics

Most reviews are short and stay a single chunk; only long ones get split (250-token windows, 18% overlap).

In [3]:
from collections import Counter

reviews = load_reviews()
tokenizer = get_model().tokenizer
chunks_per_review = [len(chunk_text(r.get('review_text', ''), tokenizer)) for r in reviews]
total_chunks = sum(chunks_per_review)

print('Reviews ingested:         ', len(reviews))
print('Total chunks embedded:    ', total_chunks)
print('Reviews split into >1:    ', sum(1 for c in chunks_per_review if c > 1))
print('Chunks-per-review counts: ', dict(sorted(Counter(chunks_per_review).items())))
print('ChromaDB collection size: ', get_collection().count())

Reviews ingested:          45
Total chunks embedded:     45
Reviews split into >1:     0
Chunks-per-review counts:  {1: 45}
ChromaDB collection size:  45


## 3. Evaluation questions (from planning.md)

Retrieve top-k = 40 per question (the spec value); the 5 highest-scoring chunks are shown for readability. `score = 1 - cosine_distance` (higher = more similar).

In [4]:
questions = [
    'Which professors quizzes and assignments are most aligned to the tests and final exam?',
    'Which professor is the most lenient for late assignment due dates?',
    'Which professors do students prefer for taking elective classes?',
    'What do students say their worst experience is for each professor?',
    'Which professors do students prefer taking for core CS courses?',
]

for q in questions:
    print('=' * 90)
    print('Q:', q)
    hits = retrieve(q, top_k=40)
    for rank, hit in enumerate(hits[:5], 1):
        m = hit['metadata']
        prof = m.get('professor_name')
        course = m.get('course')
        rating = m.get('rating_overall')
        diff = m.get('difficulty')
        src = m.get('source')
        score = hit['score']
        snippet = hit['text'][:140].replace(chr(10), ' ')
        print(f'  {rank}. [{score:.3f}] {prof} ({course})  rating={rating} difficulty={diff} [{src}]')
        print('      ', snippet, '...')

Q: Which professors quizzes and assignments are most aligned to the tests and final exam?
  1. [0.533] Shengli Yuan (CS3324)  rating=1.0 difficulty=5.0 [RateMyProfessors]
       1st assignment requires you to know C language.  One of the student asked for help during class and his response was to search in google for ...
  2. [0.522] Emre Yilmaz (CS3304)  rating=3.0 difficulty=3.0 [RateMyProfessors]
       The one thing I want to let other students know is that if you take him he will open the quiz on the day of the exam. Including he is a last ...
  3. [0.491] Cyril Harris (CS3304)  rating=1.0 difficulty=5.0 [RateMyProfessors]
       This professor may be nice in person, but his exams were a literally a nightmare. It's mostly coding, he takes points off for no reason. I h ...
  4. [0.481] Azadeh Izadi (CS-1411)  rating=1.0 difficulty=5.0 [RateMyProfessors]
       I am taking organic chemistry right now and it is much easier compared to this INTRO class. There are labs/quizzes EVERY cl

## 4. Ad-hoc query

Edit the query string and re-run to explore retrieval interactively.

In [5]:
query = 'professor who explains AI concepts clearly and is approachable'

for rank, hit in enumerate(retrieve(query, top_k=40)[:8], 1):
    m = hit['metadata']
    prof = m.get('professor_name')
    course = m.get('course')
    score = hit['score']
    snippet = hit['text'][:160].replace(chr(10), ' ')
    print(f'{rank}. [{score:.3f}] {prof} ({course})')
    print('   ', snippet, '...')

1. [0.718] Azadeh Izadi (CS-5332)


    The Introduction to AI class was an excellent learning experience. The professor explained complex AI concepts in a very clear and understandable way, and quest ...
2. [0.583] Subash Pakhrin (CS4317)
    Best Computer Science professor I've ever encountered. He is the kindest and most patient professor in and outside of class. He is more than willing to go over  ...
3. [0.516] Ting Zhang (CS3304)
    good professor, and if you want a easy time of it take her class, seems to know what they're talking about in terms of projects. ...
4. [0.497] unknown (CS4315)
    Great professor. A lot of information covered but she really wants to help us understand the materials. Some coding assignments in C but really easy since she g ...
5. [0.491] Joseph Kamto (CS1411)
    Professor Kamto explains concepts clearly and breaks down difficult material step-by-step. His assignments match what he teaches in lecture, and he gives helpfu ...
6. [0.487] Joseph Kamto (CS1411)
    I would not take this 